In [1]:
import pandas as pd
import numpy as np

# Load main table and bureau
app = pd.read_csv('../data/application_train.csv')
bureau = pd.read_csv('../data/bureau.csv')

print(f"Application table: {app.shape}")
print(f"Bureau table: {bureau.shape}")
print(f"\nBureau columns:\n{bureau.columns.tolist()}")
print(f"\nBureau sample:\n{bureau.head(3)}")

Application table: (307511, 122)
Bureau table: (1716428, 17)

Bureau columns:
['SK_ID_CURR', 'SK_ID_BUREAU', 'CREDIT_ACTIVE', 'CREDIT_CURRENCY', 'DAYS_CREDIT', 'CREDIT_DAY_OVERDUE', 'DAYS_CREDIT_ENDDATE', 'DAYS_ENDDATE_FACT', 'AMT_CREDIT_MAX_OVERDUE', 'CNT_CREDIT_PROLONG', 'AMT_CREDIT_SUM', 'AMT_CREDIT_SUM_DEBT', 'AMT_CREDIT_SUM_LIMIT', 'AMT_CREDIT_SUM_OVERDUE', 'CREDIT_TYPE', 'DAYS_CREDIT_UPDATE', 'AMT_ANNUITY']

Bureau sample:
   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0             -153.0   
1                   0               1075.0                NaN   
2                   0                528.0                NaN   

   AMT_CR

In [2]:
# Aggregate bureau data by applicant
bureau_agg = bureau.groupby('SK_ID_CURR').agg(
    # Count of credits
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    
    # Active vs closed credits
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    
    # Overdue information
    BUREAU_MAX_OVERDUE_DAYS=('CREDIT_DAY_OVERDUE', 'max'),
    BUREAU_MEAN_OVERDUE_DAYS=('CREDIT_DAY_OVERDUE', 'mean'),
    BUREAU_MAX_AMT_OVERDUE=('AMT_CREDIT_MAX_OVERDUE', 'max'),
    
    # Debt information
    BUREAU_TOTAL_DEBT=('AMT_CREDIT_SUM_DEBT', 'sum'),
    BUREAU_MEAN_DEBT=('AMT_CREDIT_SUM_DEBT', 'mean'),
    
    # Credit prolongations (extensions - sign of financial stress)
    BUREAU_TOTAL_PROLONGATIONS=('CNT_CREDIT_PROLONG', 'sum'),
    
    # Total credit amount
    BUREAU_TOTAL_CREDIT=('AMT_CREDIT_SUM', 'sum'),
    
    # Most recent credit (closest to 0 = most recent)
    BUREAU_MOST_RECENT_CREDIT=('DAYS_CREDIT', 'max')
).reset_index()

print(f"Bureau aggregated shape: {bureau_agg.shape}")
print(f"\nSample:\n{bureau_agg.head(3)}")

Bureau aggregated shape: (305811, 12)

Sample:
   SK_ID_CURR  BUREAU_LOAN_COUNT  BUREAU_ACTIVE_COUNT  BUREAU_CLOSED_COUNT  \
0      100001                  7                    3                    4   
1      100002                  8                    2                    6   
2      100003                  4                    1                    3   

   BUREAU_MAX_OVERDUE_DAYS  BUREAU_MEAN_OVERDUE_DAYS  BUREAU_MAX_AMT_OVERDUE  \
0                        0                       0.0                     NaN   
1                        0                       0.0                5043.645   
2                        0                       0.0                   0.000   

   BUREAU_TOTAL_DEBT  BUREAU_MEAN_DEBT  BUREAU_TOTAL_PROLONGATIONS  \
0           596686.5      85240.928571                           0   
1           245781.0      49156.200000                           0   
2                0.0          0.000000                           0   

   BUREAU_TOTAL_CREDIT  BUREAU_MOST_RE

In [3]:
# Join bureau aggregates to main table
app_bureau = app.merge(bureau_agg, on='SK_ID_CURR', how='left')

print(f"Shape after joining bureau: {app_bureau.shape}")
print(f"New bureau columns added: {bureau_agg.shape[1] - 1}")
print(f"\nMissing values in bureau columns (applicants with no bureau history):")
bureau_cols = [c for c in bureau_agg.columns if c != 'SK_ID_CURR']
print(app_bureau[bureau_cols].isnull().sum())

Shape after joining bureau: (307511, 133)
New bureau columns added: 11

Missing values in bureau columns (applicants with no bureau history):
BUREAU_LOAN_COUNT              44020
BUREAU_ACTIVE_COUNT            44020
BUREAU_CLOSED_COUNT            44020
BUREAU_MAX_OVERDUE_DAYS        44020
BUREAU_MEAN_OVERDUE_DAYS       44020
BUREAU_MAX_AMT_OVERDUE        123625
BUREAU_TOTAL_DEBT              44020
BUREAU_MEAN_DEBT               51380
BUREAU_TOTAL_PROLONGATIONS     44020
BUREAU_TOTAL_CREDIT            44020
BUREAU_MOST_RECENT_CREDIT      44020
dtype: int64


In [4]:
# For applicants with no bureau history - fill with 0

bureau_cols = [c for c in bureau_agg.columns if c != 'SK_ID_CURR']
app_bureau[bureau_cols] = app_bureau[bureau_cols].fillna(0)

# Create a flag for applicants with no bureau history
app_bureau['BUREAU_NO_HISTORY'] = (app_bureau['BUREAU_LOAN_COUNT'] == 0).astype(int)

print(f"Applicants with no bureau history: {app_bureau['BUREAU_NO_HISTORY'].sum()}")
print(f"\nMissing values remaining: {app_bureau[bureau_cols].isnull().sum().sum()}")

# Check default rate for no-history vs history applicants
print(f"\nDefault rate - no bureau history: {app_bureau[app_bureau['BUREAU_NO_HISTORY']==1]['TARGET'].mean()*100:.2f}%")
print(f"Default rate - has bureau history: {app_bureau[app_bureau['BUREAU_NO_HISTORY']==0]['TARGET'].mean()*100:.2f}%")

C:\Users\User\AppData\Local\Temp\ipykernel_12024\3346350273.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  app_bureau['BUREAU_NO_HISTORY'] = (app_bureau['BUREAU_LOAN_COUNT'] == 0).astype(int)


Applicants with no bureau history: 44020

Missing values remaining: 0

Default rate - no bureau history: 10.12%
Default rate - has bureau history: 7.73%


In [5]:
# Defragment
app_bureau = app_bureau.copy()

print(f"Final shape: {app_bureau.shape}")
print(f"Columns added from bureau: {app_bureau.shape[1] - app.shape[1]}")

Final shape: (307511, 134)
Columns added from bureau: 12


In [6]:
# Load previous applications
prev = pd.read_csv('../data/previous_application.csv')

print(f"Previous applications shape: {prev.shape}")
print(f"\nColumns:\n{prev.columns.tolist()}")
print(f"\nSample:\n{prev.head(3)}")

Previous applications shape: (1670214, 37)

Columns:
['SK_ID_PREV', 'SK_ID_CURR', 'NAME_CONTRACT_TYPE', 'AMT_ANNUITY', 'AMT_APPLICATION', 'AMT_CREDIT', 'AMT_DOWN_PAYMENT', 'AMT_GOODS_PRICE', 'WEEKDAY_APPR_PROCESS_START', 'HOUR_APPR_PROCESS_START', 'FLAG_LAST_APPL_PER_CONTRACT', 'NFLAG_LAST_APPL_IN_DAY', 'RATE_DOWN_PAYMENT', 'RATE_INTEREST_PRIMARY', 'RATE_INTEREST_PRIVILEGED', 'NAME_CASH_LOAN_PURPOSE', 'NAME_CONTRACT_STATUS', 'DAYS_DECISION', 'NAME_PAYMENT_TYPE', 'CODE_REJECT_REASON', 'NAME_TYPE_SUITE', 'NAME_CLIENT_TYPE', 'NAME_GOODS_CATEGORY', 'NAME_PORTFOLIO', 'NAME_PRODUCT_TYPE', 'CHANNEL_TYPE', 'SELLERPLACE_AREA', 'NAME_SELLER_INDUSTRY', 'CNT_PAYMENT', 'NAME_YIELD_GROUP', 'PRODUCT_COMBINATION', 'DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION', 'NFLAG_INSURED_ON_APPROVAL']

Sample:
   SK_ID_PREV  SK_ID_CURR NAME_CONTRACT_TYPE  AMT_ANNUITY  AMT_APPLICATION  \
0     2030495      271877     Consumer loans     1730.430          171

In [8]:
# Fix 365243 anomaly in previous applications
days_cols = ['DAYS_FIRST_DRAWING', 'DAYS_FIRST_DUE', 
             'DAYS_LAST_DUE_1ST_VERSION', 'DAYS_LAST_DUE', 'DAYS_TERMINATION']
prev[days_cols] = prev[days_cols].replace(365243, np.nan)

# Aggregate previous applications by applicant
prev_agg = prev.groupby('SK_ID_CURR').agg(
    # Count of previous applications
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    
    # Approval/rejection counts
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    PREV_CANCELLED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Cancelled').sum()),
    
    # Credit vs application amount ratio (did they get what they asked for?)
    PREV_CREDIT_TO_APP_RATIO=('AMT_APPLICATION', lambda x: (prev.loc[x.index, 'AMT_CREDIT'] / x).mean()),
    
    # Most recent application
    PREV_MOST_RECENT_APP=('DAYS_DECISION', 'max'),
    
    # Average number of payments
    PREV_AVG_CNT_PAYMENT=('CNT_PAYMENT', 'mean'),
    
    # Annuity amounts
    PREV_MAX_ANNUITY=('AMT_ANNUITY', 'max'),
    PREV_MEAN_ANNUITY=('AMT_ANNUITY', 'mean'),
).reset_index()

# Approval rate
prev_agg['PREV_APPROVAL_RATE'] = prev_agg['PREV_APPROVED_COUNT'] / prev_agg['PREV_APP_COUNT']

print(f"Previous applications aggregated: {prev_agg.shape}")
print(f"\nSample:\n{prev_agg.head(3)}")

Previous applications aggregated: (338857, 11)

Sample:
   SK_ID_CURR  PREV_APP_COUNT  PREV_APPROVED_COUNT  PREV_REFUSED_COUNT  \
0      100001               1                    1                   0   
1      100002               1                    1                   0   
2      100003               3                    3                   0   

   PREV_CANCELLED_COUNT  PREV_CREDIT_TO_APP_RATIO  PREV_MOST_RECENT_APP  \
0                     0                  0.957782                 -1740   
1                     0                  1.000000                  -606   
2                     0                  1.057664                  -746   

   PREV_AVG_CNT_PAYMENT  PREV_MAX_ANNUITY  PREV_MEAN_ANNUITY  \
0                   8.0          3951.000           3951.000   
1                  24.0          9251.775           9251.775   
2                  10.0         98356.995          56553.990   

   PREV_APPROVAL_RATE  
0                 1.0  
1                 1.0  
2                

In [9]:
# Join previous applications to main table
app_bureau_prev = app_bureau.merge(prev_agg, on='SK_ID_CURR', how='left')

# Fill missing values for applicants with no previous applications
prev_cols = [c for c in prev_agg.columns if c != 'SK_ID_CURR']
app_bureau_prev[prev_cols] = app_bureau_prev[prev_cols].fillna(0)

# Create no previous application flag
app_bureau_prev['PREV_NO_HISTORY'] = (app_bureau_prev['PREV_APP_COUNT'] == 0).astype(int)

# Defragment
app_bureau_prev = app_bureau_prev.copy()

print(f"Final shape: {app_bureau_prev.shape}")
print(f"\nDefault rate - no previous app history: {app_bureau_prev[app_bureau_prev['PREV_NO_HISTORY']==1]['TARGET'].mean()*100:.2f}%")
print(f"Default rate - has previous app history: {app_bureau_prev[app_bureau_prev['PREV_NO_HISTORY']==0]['TARGET'].mean()*100:.2f}%")

Final shape: (307511, 145)

Default rate - no previous app history: 5.96%
Default rate - has previous app history: 8.19%


In [10]:
# Save enriched dataset
app_bureau_prev.to_csv('../data/application_train_enriched.csv', index=False)
print(f"Saved enriched dataset: {app_bureau_prev.shape}")

Saved enriched dataset: (307511, 145)
